# PlantAdvisor: проверка SFT-датасета v3

Набор v3 содержит 400 синтетических диалогов для консультаций по комнатным растениям: 360 train и 40 validation, по 200 примеров ухода и диагностики. Представлены 25 растений и 141 семейство сценариев.

Проверяются структура, баланс, уникальность, происхождение и отделение от контрольных вопросов. Сборка выполняется `scripts/build_sft_v3.py`, независимый структурный аудит — `scripts/audit_sft_v3.py`. Сохранённые отчёты находятся в `results/dataset/v3/`; этот ноутбук предназначен для повторного анализа и не содержит выводов выполненного сеанса.


## 1. Загрузка входных файлов

В Colab подключается Google Drive, затем входные файлы скачиваются из одного коммита публичного GitHub и проверяются по SHA-256. Данные v3 размещаются в `MyDrive/plant-advisor/data/v3/`, отчёты — в `results/dataset/v3/`. Токены и ручная загрузка JSONL не требуются.

Локально используется копия репозитория с манифестом v3. Для анализа достаточно CPU и библиотек `pandas`, `matplotlib`, `scikit-learn`, `datasets`.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None

GITHUB_REPOSITORY = "Batyaoftheyear/plant-advisor"
GITHUB_REF = "main"  # Для точного повторения можно указать полный SHA коммита.


def load_github_sync():
    """Загрузить синхронизацию из того же публичного коммита, что и данные."""
    import importlib.util
    import os
    import subprocess
    from urllib.request import Request, urlopen

    if re.fullmatch(r"[0-9a-fA-F]{40}", GITHUB_REF):
        commit = GITHUB_REF.lower()
    else:
        result = subprocess.run(
            ["git", "-c", "credential.helper=", "-c", "core.askPass=",
             "-c", "http.extraHeader=", "ls-remote", "--exit-code",
             f"https://github.com/{GITHUB_REPOSITORY}.git",
             f"refs/heads/{GITHUB_REF}", f"refs/tags/{GITHUB_REF}",
             f"refs/tags/{GITHUB_REF}^{{}}"],
            check=True, capture_output=True, text=True, timeout=60,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""},
        )
        refs = {line.split("\t", 1)[1]: line.split("\t", 1)[0]
                for line in result.stdout.splitlines() if "\t" in line}
        commit = (refs.get(f"refs/heads/{GITHUB_REF}")
                  or refs.get(f"refs/tags/{GITHUB_REF}^{{}}")
                  or refs.get(f"refs/tags/{GITHUB_REF}"))
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError("GitHub вернул некорректный SHA коммита.")
    helper_url = (f"https://raw.githubusercontent.com/{GITHUB_REPOSITORY}/"
                  f"{commit}/scripts/sync_colab_data.py")
    request = Request(helper_url, headers={"User-Agent": "PlantAdvisor-Colab-v3"})
    with urlopen(request, timeout=60) as response:
        helper_bytes = response.read(1024 * 1024)
    helper_dir = Path("/content/plant-advisor-github") / commit
    helper_dir.mkdir(parents=True, exist_ok=True)
    helper_path = helper_dir / "sync_colab_data.py"
    helper_path.write_bytes(helper_bytes)
    spec = importlib.util.spec_from_file_location("plant_advisor_sync_v3", helper_path)
    if spec is None or spec.loader is None:
        raise ImportError("Не удалось загрузить модуль синхронизации.")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module, commit


if drive is not None:
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/plant-advisor")
    sync_module, GITHUB_COMMIT = load_github_sync()
    sync_module.sync_from_github(PROJECT_DIR, repository=GITHUB_REPOSITORY,
                                revision=GITHUB_COMMIT)
else:
    PROJECT_DIR = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                        if (candidate / "data/v3/manifest.json").is_file()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError("Запустите ноутбук из копии репозитория с data/v3/manifest.json.")
    spec = importlib.util.spec_from_file_location(
        "plant_advisor_sync_v3", PROJECT_DIR / "scripts/sync_colab_data.py"
    )
    sync_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sync_module)

sync_module.verify_local(PROJECT_DIR)
DATA_DIR = PROJECT_DIR / "data/v3"
RESULTS_DIR = PROJECT_DIR / "results/dataset/v3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Версия данных: v3")
print("Каталог проекта:", PROJECT_DIR)
print("Обучающие данные:", DATA_DIR)


## 2. Загрузка и схема

Обучающие JSONL содержат по два сообщения: `user` и `assistant`. Метаданные включают ID `SFT3-*`, категорию, растение, семейство сценария, источники и назначенную выборку. Контрольные вопросы читаются из общего неизменного `data/eval_questions.jsonl`.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    """Прочитать записи JSONL с номерами строк в сообщениях об ошибках."""
    result = []
    with path.open(encoding="utf-8") as file:
        for number, line in enumerate(file, start=1):
            if line.strip():
                try:
                    result.append(json.loads(line))
                except json.JSONDecodeError as error:
                    raise ValueError(f"{path.name}, строка {number}: {error}") from error
    return result

train = read_jsonl(DATA_DIR / "train.jsonl")
validation = read_jsonl(DATA_DIR / "validation.jsonl")
metadata = read_jsonl(DATA_DIR / "sft_metadata.jsonl")
eval_questions = read_jsonl(PROJECT_DIR / "data/eval_questions.jsonl")
provenance = pd.read_csv(DATA_DIR / "sft_provenance.csv")
sources = pd.read_csv(DATA_DIR / "sources.csv")

summary = pd.DataFrame([
    {"split": "train", "examples": len(train)},
    {"split": "validation", "examples": len(validation)},
    {"split": "eval (до/после SFT)", "examples": len(eval_questions)},
])
display(summary)
print("Количество растений:", len({record["plant"] for record in metadata}))
print("Количество типов сценариев:", len({record["scenario"] for record in metadata}))

## 3. Состав набора

В train находится по 180 примеров `care` и `diagnosis`, в validation — по 20. Для каждого растения подготовлено 16 диалогов. Семейства сценариев разделены между выборками: 132 train и 9 validation, без пересечений. Рассчитываются также длины вопросов и ответов в словах.


In [ ]:
records_df = pd.DataFrame(metadata)

counts = (
    records_df.groupby(["split", "task"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=["train", "validation"])
)
display(counts)

plant_counts = records_df.groupby("plant").size().sort_values(ascending=False)
print("Число примеров на растение: минимум", plant_counts.min(), "максимум", plant_counts.max())

records_df["question_words"] = records_df["question"].str.split().str.len()
records_df["answer_words"] = records_df["answer"].str.split().str.len()
display(
    records_df.groupby(["split", "task"])[["question_words", "answer_words"]]
    .agg(["mean", "min", "max"]).round(1)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Примеры по категориям")
axes[0].set_ylabel("Количество")
axes[0].set_xlabel("Выборка")

records_df["answer_words"].hist(bins=22, ax=axes[1])
axes[1].set_title("Длина ответов")
axes[1].set_xlabel("Количество слов")
axes[1].set_ylabel("Частота")

plt.tight_layout()
fig.savefig(RESULTS_DIR / "dataset_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Проверки формата и разделения

Проверяются роли, непустые тексты, язык и обязательные разделы ответа. Затем сопоставляются метаданные, происхождение и источники, проверяются уникальность ID и вопросов, точные пересечения с тестом и разделение семейств сценариев. Эти условия проверяют подготовку данных, а не достоверность каждого совета.


In [ ]:
REQUIRED_HEADINGS = {
    "care": ["Условия", "Рекомендации"],
    "diagnosis": ["Возможные причины", "Что проверить", "Что делать"],
}
BAD_FRAGMENTS = ("iteDatabase", "SELECT * FROM", "<script", "www.google.com/search?")


def normalize(text: str) -> str:
    return " ".join(re.findall(r"\w+", text.casefold()))


def is_cyrillic(text: str) -> bool:
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return False
    cyrillic_count = sum("а" <= ch.lower() <= "я" or ch.lower() == "ё" for ch in letters)
    return cyrillic_count / len(letters) >= 0.60


def check_headings(answer: str, task: str) -> bool:
    positions = []
    for title in REQUIRED_HEADINGS[task]:
        pattern = rf"(?m)^\s*{re.escape(title)}\s*:?\s*$"
        matches = list(re.finditer(pattern, answer, flags=re.IGNORECASE))
        if len(matches) != 1:
            return False
        positions.append(matches[0])
    if [match.start() for match in positions] != sorted(match.start() for match in positions):
        return False
    for i, match in enumerate(positions):
        end = positions[i + 1].start() if i + 1 < len(positions) else len(answer)
        if not answer[match.end():end].strip():
            return False
    return True


def check_example(item: dict, task: str) -> list[str]:
    errors = []
    messages = item.get("messages")
    if not isinstance(messages, list) or len(messages) != 2:
        return ["messages must contain exactly two entries"]
    if [msg.get("role") for msg in messages] != ["user", "assistant"]:
        errors.append("roles must be user, assistant")
    question = messages[0].get("content", "")
    answer = messages[1].get("content", "")
    if not question.strip() or not answer.strip():
        errors.append("empty text")
    if not is_cyrillic(question) or not is_cyrillic(answer):
        errors.append("language check failed")
    if not check_headings(answer, task):
        errors.append("incorrect headings")
    if any(fragment.lower() in answer.lower() for fragment in BAD_FRAGMENTS):
        errors.append("unexpected text artifact")
    return errors

meta_by_split = {split: [m for m in metadata if m["split"] == split] for split in ("train", "validation")}
checks = []
for split, items in [("train", train), ("validation", validation)]:
    assert len(items) == len(meta_by_split[split]), "Количество метаданных не соответствует выборке"
    for i, (item, meta) in enumerate(zip(items, meta_by_split[split])):
        checks.append({"split": split, "id": meta["id"], "task": meta["task"],
                       "error": "; ".join(check_example(item, meta["task"]))})

check_df = pd.DataFrame(checks)
display(check_df[check_df["error"] != ""].head(20))
print("Ошибок в структуре:", (check_df["error"] != "").sum())

In [ ]:
all_examples = train + validation
questions = [example["messages"][0]["content"] for example in all_examples]
normalized = [normalize(q) for q in questions]
eval_normalized = {normalize(q["question"]) for q in eval_questions}

assert len(set(normalized)) == len(normalized), "Обнаружены повторяющиеся обучающие вопросы"
assert not (set(normalized) & eval_normalized), "Обнаружено точное пересечение с тестовыми вопросами"
assert len({m["id"] for m in metadata}) == len(all_examples), "Повторяющиеся ID"

provenance_ids = set(provenance["id"])
source_ids = set(sources["source_id"])
assert provenance_ids == {m["id"] for m in metadata}, "Не все примеры имеют запись о происхождении"
assert all(set(m["source_ids"]) <= source_ids for m in metadata), "Неизвестный ID источника"
assert (check_df["error"] == "").all(), "Некоторые записи не прошли автоматические проверки"

print("Формат, язык, уникальность, источники и точные пересечения с eval: проверка пройдена")
train_families = {(m["task"], m["scenario"]) for m in metadata if m["split"] == "train"}
validation_families = {(m["task"], m["scenario"]) for m in metadata if m["split"] == "validation"}
assert not train_families & validation_families, "Семейства сценариев пересекаются"
assert (len(train_families), len(validation_families)) == (132, 9)
print("Семейства сценариев: 132 train, 9 validation; пересечений нет")


## 5. Близость к контрольным вопросам

Вопросы сопоставляются с тестом по косинусному сходству символьного TF–IDF с n-граммами 3–5. Для каждого примера сохраняются ближайший контрольный вопрос и значение сходства. Порог 0,85 используется для выделения пар на дополнительный просмотр; смысловую независимость эта метрика не гарантирует.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
corpus = questions + [record["question"] for record in eval_questions]
features = vectorizer.fit_transform(corpus)
similarity = (features[:len(questions)] @ features[len(questions):].T).toarray()

pairs = []
for i in range(len(questions)):
    j = int(similarity[i].argmax())
    pairs.append({
        "sft_question": questions[i],
        "closest_eval_id": eval_questions[j]["id"],
        "closest_eval_question": eval_questions[j]["question"],
        "similarity": float(similarity[i, j]),
    })

similarities = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
display(similarities.head(8))
print("Пары с сходством >= 0.85:", (similarities["similarity"] >= 0.85).sum())
similarities.to_csv(RESULTS_DIR / "eval_similarity_audit.csv", index=False)

## 6. Выборка для просмотра

Выводятся примеры с исходными вопросами, ответами и ссылками. Ячейка создаёт отдельную форму просмотра с пустыми полями оценки. Предоставленная форма `manual_review_v3.csv` на 30 записей сохраняется отдельно и не заменяется этим расчётом.


In [ ]:
review = (
    records_df.groupby("task", group_keys=False)
    .sample(n=12, random_state=42)
    .sort_values(["task", "id"])
)

for _, record in review.head(6).iterrows():
    print("=" * 85)
    print(record["id"], "|", record["task"], "|", record["plant"])
    print("Вопрос:", record["question"])
    print("Ответ:", record["answer"], sep="\n")
    print("Источники:", "; ".join(record["source_urls"]))

review_sheet = review[["id", "split", "task", "plant", "question", "answer"]].copy()
review_sheet["factually_correct"] = ""
review_sheet["actionable"] = ""
review_sheet["reviewer_notes"] = ""
review_sheet.to_csv(RESULTS_DIR / "manual_review_sample.csv", index=False, encoding="utf-8-sig")
print("Форма для ручной проверки:", RESULTS_DIR / "manual_review_sample.csv")

## 7. Совместимость с Datasets

Проверяется загрузка `train.jsonl` и `validation.jsonl` через `datasets.load_dataset`. Ожидаются две выборки на 360 и 40 строк с полем `messages`.


In [ ]:
try:
    from datasets import load_dataset
    dataset = load_dataset(
        "json",
        data_files={"train": str(DATA_DIR / "train.jsonl"),
                    "validation": str(DATA_DIR / "validation.jsonl")},
    )
    print(dataset)
    print("Первое сообщение:", dataset["train"][0]["messages"][0])
except ImportError:
    print("Для проверки в этом окружении установите datasets: %pip install datasets")

## 8. Сохранение аудита

Сводка проверок, распределения, график и таблица сходства записываются в каталог v3. Отчёты предыдущих вариантов и обучающие JSONL не заменяются.


In [ ]:
summary = {
    "dataset_version": "v3",
    "dataset_kind": "synthetic, curated, case-bank with plant profiles, v3",
    "total_examples": len(all_examples),
    "train_examples": len(train),
    "validation_examples": len(validation),
    "task_counts": counts.to_dict(orient="index"),
    "plant_count": records_df["plant"].nunique(),
    "scenario_count": records_df["scenario"].nunique(),
    "format_errors": int((check_df["error"] != "").sum()),
    "exact_eval_overlaps": int(len(set(normalized) & eval_normalized)),
    "high_similarity_pairs_threshold_0_85": int((similarities["similarity"] >= 0.85).sum()),
    "validation_method": "unseen scenario families; no shared (task, scenario) between splits",
    "limitations": [
        "Synthetic examples are not an expert-verified gold standard.",
        "The fact references describe general growing principles, not proof for every sentence.",
        "Template reuse can reduce linguistic diversity.",
    ],
}
with (RESULTS_DIR / "dataset_audit.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2, default=str)

records_df.groupby(["split", "task"]).size().rename("count").reset_index().to_csv(
    RESULTS_DIR / "split_distribution.csv", index=False
)

print("Таблицы, проверочная выборка и графики сохранены в:", RESULTS_DIR)
print("Train:", DATA_DIR / "train.jsonl")
print("Validation:", DATA_DIR / "validation.jsonl")

## Итог

Автоматический аудит опубликованных файлов подтверждает 400 уникальных пар, 141 семейство сценариев, отсутствие пересечения семейств между train и validation и отсутствие точного совпадения с 30 контрольными вопросами. Полная независимая предметная проверка не проводилась.

Результаты обучения на этом наборе и сравнение с исходной моделью сохранены в `03_training_and_evaluation_v3.ipynb`.
